# Reconciliation Dashboard Notebook\nGenerates migration reconciliation metrics for source vs target and operational quality controls.

In [ ]:
from clinical_lakehouse.reconciliation.framework import (\n    assemble_reconciliation_report,\n    change_type_counts,\n    duplicate_count,\n    match_accuracy,\n    rejected_record_count,\n    runtime_metric_df,\n    source_vs_target_record_count,\n    source_vs_target_total,\n    utc_now,\n)

In [ ]:
load_start = utc_now()\n\n# Source tables\nsrc_contract = spark.table("clinical_migration_dev.bronze.relisource_contract_raw")\nsrc_invoice = spark.table("clinical_migration_dev.bronze.gp_invoice_raw")\n\n# Target tables\ntgt_contract = spark.table("clinical_migration_dev.gold.gold_netsuite_contract_master")\ntgt_invoice = spark.table("clinical_migration_dev.gold.gold_netsuite_invoice_header")\ntgt_open_ar = spark.table("clinical_migration_dev.gold.gold_netsuite_open_ar")\n\n# Supporting quality/entity tables\nrejected_union = spark.table("clinical_migration_dev.silver_quarantine.silver_study_rejected")\nentity_crosswalk = spark.table("clinical_migration_dev.silver.silver_entity_crosswalk")\ndelta_changes = spark.table("clinical_migration_dev.audit.latest_delta_changes")

In [ ]:
record_count_metrics = source_vs_target_record_count(src_invoice, tgt_invoice, metric_group="record_count")\ncontract_value_metrics = source_vs_target_total(src_contract, "contract_value", tgt_contract, "contract_value", metric_group="contract_value")\ninvoice_total_metrics = source_vs_target_total(src_invoice, "invoice_amount", tgt_invoice, "invoice_total", metric_group="invoice_total")\nopen_ar_metrics = source_vs_target_total(src_invoice, "outstanding_amount", tgt_open_ar, "amount_due", metric_group="open_ar")\nrejected_metrics = rejected_record_count(rejected_union)\nduplicate_metrics = duplicate_count(src_invoice, key_cols=["invoice_id"], metric_group="quality")\nmatch_metrics = match_accuracy(entity_crosswalk, confidence_col="overall_match_confidence", threshold=0.97)\nchange_metrics = change_type_counts(delta_changes, metric_group="delta_load")

In [ ]:
load_end = utc_now()\nruntime_metrics = runtime_metric_df(spark, load_start, load_end)\n\nreport = assemble_reconciliation_report(\n    record_count_metrics,\n    contract_value_metrics,\n    invoice_total_metrics,\n    open_ar_metrics,\n    rejected_metrics,\n    duplicate_metrics,\n    match_metrics,\n    runtime_metrics,\n    change_metrics,\n)\ndisplay(report.orderBy("metric_group", "metric_name"))

In [ ]:
report.write.format("delta").mode("overwrite").saveAsTable("clinical_migration_dev.gold.gold_reconciliation_dashboard")